In [1]:
from pathlib import Path

import polars as pl
import numpy as np
from sklearn.model_selection import StratifiedKFold

ROOT = Path(
    "/heinz-georgenas/users/mingzhul/"
    "Simultaneous-EHR/yaib/data/mortality24"
)

SEED = 0

DATASETS = {
    "HiRID": ["hirid", "hirid_15min", "hirid_5min"],
    "MIMIC-IV": ["miiv", "miiv_15min", "miiv_5min"],
}


def get_cohort(dataset):
    outcome = pl.read_parquet(
        ROOT / dataset / "outc.parquet",
        columns=["stay_id", "label"],
    )

    # Reproduce YAIB's label aggregation for splitting.
    return (
        outcome.group_by("stay_id")
        .agg(pl.col("label").max())
        .sort("stay_id")
    )


def get_test_ids(cohort, seed):
    stays = cohort["stay_id"].to_numpy()
    labels = cohort["label"].to_numpy()

    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )

    # Repetition 0 = first outer split.
    _, test_idx = next(splitter.split(stays, labels))
    return set(stays[test_idx].tolist())


for hospital, datasets in DATASETS.items():
    print(f"\n=== {hospital} ===")

    reference = get_cohort(datasets[0])
    reference_test = get_test_ids(reference, SEED)

    for dataset in datasets:
        cohort = get_cohort(dataset)
        test_ids = get_test_ids(cohort, SEED)

        same_stays = set(cohort["stay_id"]) == set(reference["stay_id"])
        same_labels = cohort.equals(reference)
        same_test = test_ids == reference_test

        print(
            f"{dataset:15s} "
            f"stays={len(cohort):6d} "
            f"test={len(test_ids):6d} "
            f"same_stays={same_stays} "
            f"same_labels={same_labels} "
            f"same_test={same_test}"
        )

        assert same_stays, f"Stay IDs differ: {dataset}"
        assert same_labels, f"Labels differ: {dataset}"
        assert same_test, f"Test IDs differ: {dataset}"


=== HiRID ===
hirid           stays= 12859 test=  2572 same_stays=True same_labels=True same_test=True
hirid_15min     stays= 12859 test=  2572 same_stays=True same_labels=True same_test=True
hirid_5min      stays= 12859 test=  2572 same_stays=True same_labels=True same_test=True

=== MIMIC-IV ===
miiv            stays= 49523 test=  9905 same_stays=True same_labels=True same_test=True
miiv_15min      stays= 49523 test=  9905 same_stays=True same_labels=True same_test=True
miiv_5min       stays= 49523 test=  9905 same_stays=True same_labels=True same_test=True
